# Your First Vector RAG Application: Cat Health Assistant

In this notebook, we will build a dense vector retrieval application using **LangChain v1**, **OpenAI embeddings**, and **Qdrant** as an in-memory vector database.

The goal is to understand the core RAG loop:

1. Load a cat health guideline PDF
2. Split it into smaller chunks
3. Embed those chunks
4. Store the embeddings in Qdrant
5. Retrieve relevant chunks for a question
6. Generate an answer grounded in the retrieved context

> Note: This notebook expects Python 3.12 and uses uv for dependency management.

> Note: This is a vector RAG lesson, not a veterinary care tool. The assistant should answer from the PDF and point users to a veterinarian for diagnosis, treatment, medication, or urgent care decisions.

## Table of Contents

- Task 1: Environment Setup
- Task 2: Embedding Similarity Primer
- Task 3: Documents - Loading the Cat Health Guideline PDF
- Task 4: Chunking the Documents
- Task 5: Embeddings and Qdrant
- Task 6: Retrieval with Scores
- Task 7: Retrieval Augmented Generation
- Activity: Tune Retrieval Quality

## Task 1: Environment Setup

From the `01_Dense_Vector_Retrieval` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain v1 separates integrations into partner packages. We will use:

- `langchain_community` for PDF loading
- `langchain_text_splitters` for chunking
- `langchain_openai` for chat and embedding models
- `langchain_qdrant` for the Qdrant vector store

In [1]:
from pathlib import Path
from math import sqrt
from getpass import getpass
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

/var/folders/nn/sjkkz9tj6nn1g3sq5w_dm_zw0000gn/T/ipykernel_86152/4147450701.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


### OpenAI API Key

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

In [3]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

KeyboardInterrupt: Interrupted by user

In [5]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_API_KEY_HERE"

In [6]:
bool(os.environ.get("OPENAI_API_KEY"))

True

## Task 2: Embedding Similarity Primer

Before we load a full PDF, let's make dense vector retrieval less mysterious.

An embedding model turns text into a list of numbers. Texts with related meaning should land closer together in that vector space.

A common way to score closeness is **cosine similarity**:

```text
cosine_similarity(a, b) = dot_product(a, b) / (length(a) * length(b))
```

The intuition: if two vectors point in a similar direction, their cosine similarity is higher. Vector databases like Qdrant use this same idea, but at a much larger scale.

In [7]:
embedding_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embedding_model)

example_texts = [
    "king",
    "queen",
    "banana",
    "cat",
    "veterinarian",
    "cat health guidelines",
]

example_vectors = dict(zip(example_texts, embeddings.embed_documents(example_texts)))


def cosine_similarity(vector_a: list[float], vector_b: list[float]) -> float:
    dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
    length_a = sqrt(sum(a * a for a in vector_a))
    length_b = sqrt(sum(b * b for b in vector_b))
    return dot_product / (length_a * length_b)


comparison_pairs = [
    ("king", "queen"),
    ("king", "banana"),
    ("cat", "veterinarian"),
    ("cat", "cat health guidelines"),
]

for left, right in comparison_pairs:
    score = cosine_similarity(example_vectors[left], example_vectors[right])
    print(f"{left:>22} <> {right:<22} score={score:.3f}")

                  king <> queen                  score=0.591
                  king <> banana                 score=0.310
                   cat <> veterinarian           score=0.356
                   cat <> cat health guidelines  score=0.496


A few important notes:

- The score is useful for ranking, not as an absolute truth about meaning.
- Different embedding models can produce different scores.
- In RAG, we embed each document chunk once, then embed the user's query and search for the nearest chunk vectors.

That is the retrieval part of RAG.

## Task 3: Documents

LangChain represents loaded text as `Document` objects. A `Document` has:

- `page_content`: the text
- `metadata`: information such as source file and page number

We will load one `Document` per PDF page, then split those pages into smaller chunks.

### Course PDF

This notebook uses the bundled cat health guideline PDF at:

```text
01_Dense_Vector_Retrieval/data/cat_health_guidelines.pdf
```

The next cell checks that the course material is present before we start loading pages.

In [8]:
pdf_path = Path("data/cat_health_guidelines.pdf")

if not pdf_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health guideline PDF at: {pdf_path.resolve()}\n"
        "The bundled course PDF is missing from this copy of the materials."
    )

### Load the PDF

`PyPDFLoader` extracts text from text-based PDFs. If the PDF is scanned images, this loader may return little or no text, and OCR would be needed.

In [9]:
loader = PyPDFLoader(str(pdf_path))
pages = loader.load()

for page in pages:
    page.metadata["source"] = pdf_path.name
    page.metadata["document_type"] = "cat_health_guideline"

pages = [page for page in pages if page.page_content.strip()]

if not pages:
    raise ValueError(
        "The PDF loaded, but no extractable text was found. "
        "This usually means the PDF is scanned and needs OCR first."
    )

print(f"Loaded {len(pages)} text-containing PDF pages.")

Loaded 22 text-containing PDF pages.


In [10]:
print(pages[0].page_content[:750])
print("\nMetadata:", pages[0].metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177/1098612X21993657) and theJournal of the American Animal Hospital
Association(volume 57, issue 2, pages 51–72, DOI: 10.5326/JAAHA-MS-7189). A

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'so

#### ❓Question #1

Why is metadata important for a RAG application?

##### ✅ Answer:

Metadata is important in a RAG application because it improves the quality of output by helping the system retrieve the right information, ensure trustworthy answers, and help with latency. Retrieval quality depends on filtering and context selection, and metadata helps with that. 

## Task 4: Chunking the Documents

A full PDF page can be too large or too mixed-topic for high-quality retrieval. We split pages into overlapping chunks so each chunk has enough local context but is still focused.

Here we will start with chunks of 1,000 characters and 200 characters of overlap. The chunk size controls how much text each vector represents; the overlap keeps nearby context from being lost at chunk boundaries.

`RecursiveCharacterTextSplitter` tries to split on natural boundaries first, such as paragraphs and line breaks, before falling back to smaller separators.

In [44]:
chunk_size = 500
chunk_overlap = 200

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
)

splits = text_splitter.split_documents(pages)

print(f"Split {len(pages)} pages into {len(splits)} chunks.")
print(f"Chunk size: {chunk_size} characters")
print(f"Chunk overlap: {chunk_overlap} characters")

Split 22 pages into 333 chunks.
Chunk size: 500 characters
Chunk overlap: 200 characters


In [46]:
sample_chunk = splits[0]
print(sample_chunk.page_content[:750])
print("\nMetadata:", sample_chunk.metadata)

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA

Metadata: {'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'cat_health_guideline', 'start_index': 0}


#### ❓Question #2

What tradeoff do we make when choosing chunk size and chunk overlap?

##### ✅ Answer:

Key tradeoff decisions made when choosing chunk size and overlap include retrieval quality, precision, context preservation, cost and latency. 

Use the [Chunk Visualizer](https://chunkviz.up.railway.app/) to experiment with different chunk sizes and overlaps and see how the text boundaries change.

## Task 5: Embeddings and Qdrant

Now we apply the same embedding idea to every chunk from the PDF. Qdrant stores those vectors and lets us search for chunks that are close to a query in embedding space.

We already created an OpenAI embedding model in the primer above. The Qdrant collection name is just a label for the set of vectors we are creating.

For this notebook, Qdrant runs in memory with `location=":memory:"`. That means no Docker, no Qdrant Cloud account, and no persistence after the notebook kernel stops.

In [47]:
collection_name = "cat_health_guidelines"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
    force_recreate=True,
)

print(f"Embedded chunks with: {embedding_model}")
print(f"Built in-memory Qdrant collection: {collection_name}")

Embedded chunks with: text-embedding-3-small
Built in-memory Qdrant collection: cat_health_guidelines


## Task 6: Retrieval with Scores

Before we generate answers, we should inspect retrieval directly. If retrieval returns poor context, the final answer will usually be poor too.

The value `k` controls how many chunks the retriever returns. A larger `k` gives the model more context, but it can also add noise. We will start with `k = 4` and tune it later.

Qdrant can return both the matching `Document` and a similarity score. This is the same ranking idea we saw with `king`, `queen`, and `cat`, now applied to PDF chunks.

In [34]:
def display_retrieval_results(query: str, k: int) -> list[tuple]:
    """Retrieve chunks and print a compact view of the results."""
    results = vector_store.similarity_search_with_score(query, k=k)

    for index, (doc, score) in enumerate(results, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        start_index = doc.metadata.get("start_index", "unknown")
        preview = doc.page_content[:350].replace("\n", " ")

        print(f"Source {index} | score={score:.3f} | page={page_display} | start_index={start_index}")
        print(preview)
        print("-" * 80)

    return results

In [35]:
retrieval_k = 4
retrieval_query = "What signs suggest that a cat should be seen by a veterinarian?"
retrieved_results = display_retrieval_results(retrieval_query, k=retrieval_k)

Source 1 | score=0.590 | page=10 | start_index=4373
Senior cats exhibiting new or unusual behavior should be evaluated for medical conditions. 12 Changes in litter box usage may indicate urinary tract disease, constipation, or diabetes but may also be due to reduced musculoskeletal strength, impaired balance, or onset of pain. V ocali- zation, especially nighttime waking, is a common concern and may
--------------------------------------------------------------------------------
Source 2 | score=0.563 | page=12 | start_index=1954
is noted by the owner, the kitten should be evaluated for underlying conditions such as congenital abnormalities of the lower urinary or GI tract, GI parasites, or other infectious diseases. Mature adult and senior cats may house-soil secondarily to medical or behavioral conditions. Clients should be encouraged to seek veterinary assistance promptl
--------------------------------------------------------------------------------
Source 3 | score=0.561 | page=6 

#### ❓Question #3

What does a similarity score help you understand, and what does it not prove by itself?

##### ✅ Answer:

A similarity score measures how semantically close a retrieved chunk is to a query in embedding space. It helps rank and evaluate retrieval results, but it does not provde that the information is correct, complete, authoritative or up to date. A high similarity score indicates relevant, not accuracy, truth or quality. Also worth noting that similarity scores are relative not absolute, and a function of the embedding model, data set, chunking strategy, and similarity metric. 

## Task 7: Retrieval Augmented Generation

Now we combine retrieval with generation. We will use a two-step RAG pattern:

1. Retrieve relevant chunks from Qdrant
2. Put those chunks into the prompt and ask the model to answer from the context

This is intentionally simpler than an agent. We always retrieve before answering, which makes the vector retrieval mechanics easy to inspect.

For generation, we will use `gpt-5.4-mini`.

In [36]:
chat_model = "gpt-5.4-mini"
llm = ChatOpenAI(model=chat_model)

RAG_SYSTEM_PROMPT = """You are a cat health guideline assistant in a vector RAG lesson.

Use only the provided context to answer the user's question.
If the context does not contain enough information, say: "I don't have enough information in the provided cat health guideline PDF to answer that."

Cite the retrieved sources inline using labels like [Source 1] or [Source 2].
Do not diagnose, prescribe medication, or replace a veterinarian.
For diagnosis, treatment decisions, medication questions, or urgent symptoms, recommend contacting a veterinarian.
Keep the answer concise and practical."""

RAG_USER_PROMPT = """Context:
{context}

Question: {question}

Answer from the context above."""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", RAG_USER_PROMPT),
    ]
)

rag_chain = rag_prompt | llm | StrOutputParser()

In [38]:
def format_context(scored_docs: list[tuple]) -> str:
    """Convert retrieved documents into a source-labeled context string."""
    formatted_chunks = []

    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        page_display = page + 1 if isinstance(page, int) else "unknown"
        source = doc.metadata.get("source", "unknown source")

        formatted_chunks.append(
            f"[Source {index}] {source}, page {page_display}, score {score:.3f}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(formatted_chunks)


def answer_question(question: str, k: int) -> dict:
    """Run retrieve-then-generate and return the answer plus source metadata."""
    scored_docs = vector_store.similarity_search_with_score(question, k=k)
    context = format_context(scored_docs)
    answer = rag_chain.invoke({"context": context, "question": question})

    sources = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        page = doc.metadata.get("page")
        sources.append(
            {
                "source_label": f"Source {index}",
                "file": doc.metadata.get("source"),
                "page": page + 1 if isinstance(page, int) else None,
                "start_index": doc.metadata.get("start_index"),
                "score": score,
            }
        )

    return {
        "question": question,
        "answer": answer,
        "sources": sources,
        "context": scored_docs,
    }

Before calling the model, inspect the formatted context. This is the exact text that will be inserted into the RAG prompt.

In [39]:
example_context = format_context(retrieved_results[:2])
print(example_context[:2000])

[Source 1] cat_health_guidelines.pdf, page 10, score 0.590
Senior cats exhibiting new or unusual behavior should be evaluated for
medical conditions.
12 Changes in litter box usage may indicate urinary
tract disease, constipation, or diabetes but may also be due to reduced
musculoskeletal strength, impaired balance, or onset of pain. V ocali-
zation, especially nighttime waking, is a common concern and may
represent sensory changes (declin ing hearing and vision), cognitive
dysfunction syndrome, pain, hyper thyroidism, or hypertension. V et-

[Source 2] cat_health_guidelines.pdf, page 12, score 0.563
is noted by the owner, the kitten should be evaluated for underlying
conditions such as congenital abnormalities of the lower urinary or GI
tract, GI parasites, or other infectious diseases. Mature adult and senior
cats may house-soil secondarily to medical or behavioral conditions.
Clients should be encouraged to seek veterinary assistance promptly, in
order to diagnose life-threatening c

In [40]:
answer_k = 4

result = answer_question(
    "What are signs that my cat may need veterinary attention?",
    k=answer_k,
)

print(result["answer"])
print("\nSources:")
for source in result["sources"]:
    print(source)

Signs that may mean your cat needs veterinary attention include:

- **Changes in grooming**: increased grooming may point to a skin problem; **reduced grooming** may signal illness, bladder pain, joint pain, or reduced mobility [Source 1][Source 4]
- **Changes in litter box use**: may indicate urinary tract disease, constipation, diabetes, reduced strength, balance problems, or pain [Source 2]
- **New or unusual behavior in senior cats**: should be evaluated for medical conditions [Source 2]
- **Nighttime waking or increased vocalization**: can be linked to sensory changes, cognitive dysfunction, pain, hyperthyroidism, or hypertension [Source 2]
- **Signs of pain or anxiety**: especially relevant in mature adult or senior cats [Source 3]
- **Mobility or joint issues**: a musculoskeletal exam is important because osteoarthritis is common and often underdiagnosed [Source 3][Source 4]

If you’re seeing any of these changes, it’s a good idea to contact a veterinarian for an evaluation.

So

In [41]:
for question in vibe_check_questions:
    print("Question:", question)
    display_retrieval_results(question, k=answer_k)
    print("=" * 100)

Question: What preventive care is recommended for cats?
Source 1 | score=0.647 | page=21 | start_index=7033
Prevention. Available at: https://www.cdc.gov/healthypets/index.html. 129. Stull JW , Bjorvik E, Bub J, et al. 2018 AAHA infection control, pre- vention, and biosecurity guidelines. J Am Anim Hosp Assoc 2018;54: 297–326. Available at: https://www.aaha.org/biosecurity. 2021 AAHA/AAFP Feline Life Stage Guidelines JAAHA.ORG 71
--------------------------------------------------------------------------------
Source 2 | score=0.638 | page=2 | start_index=821
alized risk assessment, preventive healthcare strategies, and treat- ment pathways that evolve as the cat matures. An evidence-guided framework for managing a cat ’s healthcare throughout its lifetime has never been more important in feline practice than it is now. Cats are the most popular pet in the United States. 1 A great anomaly in feline prac
--------------------------------------------------------------------------------
Sou

### Vibe Check Queries

Run a few questions that should be answerable from a cat health guideline PDF. Then run one question that may not be answerable and confirm the assistant says it does not have enough information.

In [42]:
vibe_check_questions = [
    "What preventive care is recommended for cats?",
    "What symptoms should make me call a veterinarian?",
    "What should I know about feeding a healthy adult cat?",
    "Can my cat help me file my taxes?",
]

for question in vibe_check_questions:
    print("Question:", question)
    print(answer_question(question, k=answer_k)["answer"])
    print("=" * 100)

Question: What preventive care is recommended for cats?
The guideline emphasizes **preventive healthcare throughout a cat’s life**, using an **evidence-guided framework** with **personalized risk assessment** and **treatment pathways that change as the cat matures** [Source 2]. It also notes that **routine parasite prevention/prophylaxis** can help reduce contamination with infectious and zoonotic agents, and that many cats may benefit from **regular broad-spectrum parasite control**, especially if they have outdoor exposure or go to places like boarding or grooming facilities [Source 4].

For more specific preventive care decisions, it’s best to **contact a veterinarian** since the right plan depends on the cat’s age, lifestyle, and risk factors.
Question: What symptoms should make me call a veterinarian?
You should call a veterinarian promptly if you notice:

- **Changes in litter box use** — this may indicate **urinary tract disease, constipation, or diabetes** [Source 2]
- **House-

#### ❓Question #4

For the vibe check queries above, did the retrieved context seem relevant before generation? Why or why not?

##### ✅ Answer:

Yes, for #1-3, and correctly identified as not for #4. 

Question 1: Mixed but usable. Sources 2 & 4 are about preventative care, source 1 is tangential, source 3 is ciation. Response is pretyy good, but not all 4 chunks highly relevant. 
Question 2: Relevant content, but scores lower. Chunks do mention symptoms, so the answer is grounded in chunks. Retrieval is good, but pdf content could be better. We dont verify medical correctness. 
Question 3: Strong retrieval. All 4 chunks relevant to topic, answer is grounded to chunks, and high scores. 
Question 4: Poor retrieval. No chunks mention taxes. System still returned something, but low scores. The answer behavior is correct (refusal) 

## 🏗️ Activity: Tune Retrieval Quality

Improve retrieval quality by changing one or more of these values:

- The chunk size
- The chunk overlap
- The retrieval `k`
- The wording of the retrieval query

Suggested workflow:

1. Pick one test question.
2. Inspect the retrieved chunks and scores.
3. Change one retrieval setting.
4. Rebuild the splitter and vector store.
5. Compare whether the retrieved chunks became more relevant.

When you are done, write down what changed and whether the final answer improved.

In [56]:
# Activity workspace
# Try retrieval tuning here.

### YOUR CODE HERE

test_question = "What symptoms should make me call a veterinarian?"

print("=== RETRIEVAL (chunk_size 500, overlap 200) ===")
display_retrieval_results(test_question, k=4)

print("=== ANSWER ===")
print(answer_question(test_question, k=4)["answer"])



=== RETRIEVAL (chunk_size 500, overlap 200) ===
Source 1 | score=0.457 | page=18 | start_index=4402
clinical signs, nutrition, water intake, behavior, environment, and lifestyle. Use of open-ended questions will encourage owners to provide the team with as much useful information as possible; closed-ended questions that lend only yes or no answers may, by contrast, provide only limited information. Often, a small change in the way a question is a
--------------------------------------------------------------------------------
Source 2 | score=0.447 | page=7 | start_index=3099
vomiting hairballs, or diarrhea are of key importance to guide di- agnostic testing. Discussion should also be held with the client about increased nocturnal activity and vocalization as well as changes in the cat ’s normal habits or activity. These may indicate cognitive dysfunction, disease-reduced mobility, pain, or reduced vision. TABLE 3 Disea
------------------------------------------------------------------

### 🏗️ Activity Notes

Testing 3 levers: 
1) query tuning, 
2) chunk size, 
3) chunk overlap. 

Testing Q2, as scores are low (0.39-0.44) even though chunks were somewhat relevant - looking to see what we can do to improve that and corresponding answers

+++++

Test 1: Query tuning 

test_question = "What symptoms should make me call a veterinarian?"
tuned_query = "What symptoms should make me call a veterinarian?" 

- Before + After 

=== BASELINE RETRIEVAL ===

Source 1 | score=0.435 | page=7 | start_index=2384

Source 2 | score=0.404 | page=10 | start_index=4793

Source 3 | score=0.400 | page=7 | start_index=3237

Source 4 | score=0.390 | page=7 | start_index=820

=== BASELINE ANSWER ===

Based on the guideline context, you should contact a veterinarian if your cat has **vomiting, vomiting hairballs, diarrhea, changes in appetite, increased urination or drinking, or changes in normal habits/activity** that seem new or persistent [Source 1]. Also watch for **increased nocturnal activity, vocalization, reduced jumping/climbing, or other behavior changes**, which may indicate underlying disease, pain, or mobility issues [Source 2][Source 3].

If the symptoms are severe, sudden, or your cat seems very unwell, contact a veterinarian promptly.


=== TUNED RETRIEVAL ===

Source 1 | score=0.688 | page=8 | start_index=0

Source 2 | score=0.648 | page=7 | start_index=3237

Source 3 | score=0.639 | page=7 | start_index=820

Source 4 | score=0.637 | page=7 | start_index=2384

=== TUNED ANSWER ===

Signs that suggest a cat should be examined by a veterinarian include changes in appetite, vomiting, vomiting hairballs, diarrhea, polyuria/polydipsia, increased nocturnal activity or vocalization, and changes in normal habits, activity, demeanor, or grooming [Source 4] [Source 2] [Source 1]. Signs of pain, anxiety, reduced mobility, reduced vision, or possible skin/dermatologic problems are also reasons for examination [Source 1] [Source 2]. If you’re concerned, contact a veterinarian for an evaluation.




Did retrieval improve? Why or why not?

Yes. Scores jumped from 0.435 → 0.688 (~58% higher). All four scores improved. (1) Query wording matched the PDF; (2) Chunk 2 is more relevant to question (3) The tuned answer better grounded to source 

But could be better. Sources 3–4 still include weaker matches. Same underlying chunks appeared in both runs.




Test 2: Chunk size 

 Query tuning re-ranks what's already in the store (same 135 chunks). To improve which chunks exist, now test how the PDF is split and re-embed with chunk size.  

 Smaller chunks (more focused)
 chunk size = 500, chunk overlap=100

 Split 22 pages into 263 chunks.
Chunk size: 500 characters
Chunk overlap: 100 characters

test_question = "What symptoms should make me call a veterinarian?"

BASELINE SAME AS ABOVE. 

=== RETRIEVAL (chunk_size 500) ===

Source 1 | score=0.436 | page=12 | start_index=1954

Source 2 | score=0.429 | page=10 | start_index=4373

Source 3 | score=0.418 | page=7 | start_index=2364

Source 4 | score=0.410 | page=10 | start_index=4793

=== ANSWER ===

You should contact a veterinarian promptly if your cat has new or unusual behavior, especially in a senior cat [Source 2]. Signs that may need veterinary evaluation include:

- Changes in litter box usage, which can indicate urinary tract disease, constipation, diabetes, pain, balance problems, or weakness [Source 2]
- House-soiling or other litter box problems, since this can be linked to medical or behavioral issues and may include serious conditions like urinary tract blockage [Source 1]
- Vocalization, especially if your cat wakes at night, because it may reflect sensory changes, cognitive dysfunction, pain, hyperthyroidism, or hypertension [Source 2]
- Vomiting, vomiting hairballs, diarrhea, or chronic GI signs [Source 3]
- Weight loss or other weight changes that seem concerning [Source 3]

If you notice these symptoms, a veterinarian should evaluate your cat; some causes can be serious [Source 1][Source 2].




How did the retrieval improve? (compared to baseline, no changes)

Topics are now more relevant. Scores did not improve like test 1 with tuned query (which is understood since query was closer to word in pdf). Answers are grounded in chunks. 

 

Test 3. Overlap 

chunk size = 500, overlap = 200 (instead of 100)

Split 22 pages into 333 chunks.
Chunk size: 500 characters
Chunk overlap: 200 characters


BASELINE SAME AS ABOVE. 

=== RETRIEVAL (chunk_size 500, overlap 200) ===

Source 1 | score=0.457 | page=18 | start_index=4402

Source 2 | score=0.447 | page=7 | start_index=3099

Source 3 | score=0.413 | page=10 | start_index=4588

Source 4 | score=0.411 | page=13 | start_index=652


=== ANSWER ===

Symptoms that should prompt a call to a veterinarian include vomiting hairballs, diarrhea, increased nocturnal activity or vocalization, changes in normal habits or activity, elimination issues, and defecating outside the litter box [Source 2][Source 3][Source 4]. In senior cats, these signs may be associated with pain, mobility problems, sensory changes, cognitive dysfunction, or other disease, so a veterinary evaluation is recommended [Source 2][Source 3][Source 4].

If your cat is showing any of these symptoms, contact a veterinarian for guidance [Source 4].



Assessment (compared to baseline, no changes):
Answer looks good. Grounded in Sources 2–4; response more concise and shorter than Test 2.  Incrased from 263 to 333 chunks. For the limited improvement, probably not worthwhile. 